# Data Preprocessing

## 1. Load Datase and Inspect

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

In [120]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.append(
    str(PROJECT_ROOT)
)

In [121]:
from src.data.ingestion import load_raw_data
from src.data.preprocessing import preprocess_data
from src.data.validation import validate_dataset

In [122]:
#Load the dataset using your new ingestion module 
RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "household_power_consumption.txt"
)

df = load_raw_data(
    RAW_DATA_PATH
)

print(
    "Raw dataset shape:",
    df.shape
)

Raw dataset shape: (2075259, 9)


In [11]:
# Find the project root correctly
PROJECT_ROOT = Path.cwd().parent

print("Project root:")
print(PROJECT_ROOT)

Project root:
c:\Users\sayum\Desktop\smartenergy-ai


In [12]:
# Define the path to the raw data file
RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "household_power_consumption.txt"
)

print("Raw data path:")
print(RAW_DATA_PATH)

Raw data path:
c:\Users\sayum\Desktop\smartenergy-ai\data\raw\household_power_consumption.txt


In [13]:
# Check that the file exists
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {RAW_DATA_PATH}"
    )

print("Dataset found successfully.")

Dataset found successfully.


In [14]:
# Load the raw dataset
df = pd.read_csv(
    RAW_DATA_PATH,
    sep=";",
    low_memory=False
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [15]:
# Inspect the first records

display(df.head()) # Inspect the first records
display(df.tail()) # Inspect the last records
display(df.head().T) # Transpose the first few records for better readability

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
2075254,26/11/2010,20:58:00,0.946,0.000,240.430,4.000,0.000,0.000,0.0
2075255,26/11/2010,20:59:00,0.944,0.000,240.000,4.000,0.000,0.000,0.0
2075256,26/11/2010,21:00:00,0.938,0.000,239.820,3.800,0.000,0.000,0.0
2075257,26/11/2010,21:01:00,0.934,0.000,239.700,3.800,0.000,0.000,0.0
2075258,26/11/2010,21:02:00,0.932,0.000,239.550,3.800,0.000,0.000,0.0


,0,1,2,3,4
Date,16/12/2006,16/12/2006,16/12/2006,16/12/2006,16/12/2006
Time,17:24:00,17:25:00,17:26:00,17:27:00,17:28:00
Global_active_power,4.216,5.360,5.374,5.388,3.666
Global_reactive_power,0.418,0.436,0.498,0.502,0.528
Voltage,234.840,233.630,233.290,233.740,235.680
Global_intensity,18.400,23.000,23.000,23.000,15.800
Sub_metering_1,0.000,0.000,0.000,0.000,0.000
Sub_metering_2,1.000,1.000,2.000,1.000,1.000
Sub_metering_3,17.0,16.0,17.0,17.0,17.0


In [16]:
# Check the dataset dimensions

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 2075259
Number of columns: 9


In [17]:
print("Columns:") 
for column in df.columns:
    print(f" - {column}")

Columns:
 - Date
 - Time
 - Global_active_power
 - Global_reactive_power
 - Voltage
 - Global_intensity
 - Sub_metering_1
 - Sub_metering_2
 - Sub_metering_3


In [18]:
# Check data types
print(df.dtypes)  # The raw dataset needs preprocessing.

Date                         str
Time                         str
Global_active_power          str
Global_reactive_power        str
Voltage                      str
Global_intensity             str
Sub_metering_1               str
Sub_metering_2               str
Sub_metering_3           float64
dtype: object


## 2. MISSING VALUES

In [19]:
# Missing-value report
missing_count = df.isna().sum()

missing_percentage = (
    df.isna().mean() * 100
).round(2)

missing_report = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": missing_percentage
})

display(
    missing_report.sort_values(
        "missing_percentage",
        ascending=False
    )
)

,missing_count,missing_percentage
Sub_metering_3,25979,1.25
Date,0,0.00
Time,0,0.00
Global_reactive_power,0,0.00
Global_active_power,0,0.00
Voltage,0,0.00
Global_intensity,0,0.00
Sub_metering_1,0,0.00
Sub_metering_2,0,0.00


In [20]:
# Check duplicate rows
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


## 3. Create timestamp

In [21]:
df["timestamp"] = pd.to_datetime( 
    df["Date"].astype(str) + " " + df["Time"].astype(str),
    errors="coerce"
)

C:\Users\sayum\AppData\Local\Temp\ipykernel_15584\2845618200.py:1: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["timestamp"] = pd.to_datetime(


In [22]:
# Display the first 10 rows of the DataFrame with the new timestamp column
display(
    df[
        ["Date", "Time", "timestamp"]
    ].head(10)
)

,Date,Time,timestamp
0,16/12/2006,17:24:00,2006-12-16 17:24:00
1,16/12/2006,17:25:00,2006-12-16 17:25:00
2,16/12/2006,17:26:00,2006-12-16 17:26:00
3,16/12/2006,17:27:00,2006-12-16 17:27:00
4,16/12/2006,17:28:00,2006-12-16 17:28:00
5,16/12/2006,17:29:00,2006-12-16 17:29:00
6,16/12/2006,17:30:00,2006-12-16 17:30:00
7,16/12/2006,17:31:00,2006-12-16 17:31:00
8,16/12/2006,17:32:00,2006-12-16 17:32:00
9,16/12/2006,17:33:00,2006-12-16 17:33:00


## 4. Check invalid timestamps

In [23]:

invalid_timestamp_count = df["timestamp"].isna().sum()

print(
    "Invalid timestamps:",
    invalid_timestamp_count
)

Invalid timestamps: 0


## 5. Convert measurement columns

In [24]:
# List of columns to convert to numeric type
NUMERIC_COLUMNS = [ 
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

In [25]:
# Convert columns to numeric type
for column in NUMERIC_COLUMNS: 
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )
print(df[NUMERIC_COLUMNS].dtypes)

Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object


## 6. Check missing values again

In [26]:
missing_count = df.isna().sum()

missing_percentage = (
    df.isna().mean() * 100
).round(2)

missing_report = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": missing_percentage
})

display(
    missing_report.sort_values(
        "missing_count",
        ascending=False
    )
)

,missing_count,missing_percentage
Global_reactive_power,25979,1.25
Global_active_power,25979,1.25
Sub_metering_1,25979,1.25
Sub_metering_2,25979,1.25
Voltage,25979,1.25
Global_intensity,25979,1.25
Sub_metering_3,25979,1.25
Date,0,0.00
Time,0,0.00
timestamp,0,0.00


## 7. Check numeric statistics

In [27]:
display(
    df[NUMERIC_COLUMNS].describe().T
)

,count,mean,std,min,25%,50%,75%,max
Global_active_power,2049280.0,1.091615,1.057294,0.076,0.308,0.602,1.528,11.122
Global_reactive_power,2049280.0,0.123714,0.112722,0.000,0.048,0.100,0.194,1.390
Voltage,2049280.0,240.839858,3.239987,223.200,238.990,241.010,242.890,254.150
Global_intensity,2049280.0,4.627759,4.444396,0.200,1.400,2.600,6.400,48.400
Sub_metering_1,2049280.0,1.121923,6.153031,0.000,0.000,0.000,0.000,88.000
Sub_metering_2,2049280.0,1.298520,5.822026,0.000,0.000,0.000,1.000,80.000
Sub_metering_3,2049280.0,6.458447,8.437154,0.000,0.000,1.000,17.000,31.000


## 8. Sort chronologically

In [28]:
# Sort the DataFrame by timestamp
df = df.sort_values( 
    "timestamp"
).reset_index(drop=True)

In [29]:
# Check if the timestamp column is sorted
print(
    "Timestamp sorted:",
    df["timestamp"].is_monotonic_increasing
)

Timestamp sorted: True


In [30]:
# Check duplicate timestamps
timestamp_duplicate_count = (
    df["timestamp"].duplicated().sum()
)

print(
    "Duplicate timestamps:",
    timestamp_duplicate_count
)

Duplicate timestamps: 0


## 9. Check the time range

In [31]:
print(
    "First timestamp:",
    df["timestamp"].min()
)

print(
    "Last timestamp:",
    df["timestamp"].max()
)

First timestamp: 2006-12-16 17:24:00
Last timestamp: 2010-11-26 21:02:00


In [32]:
# Check whether measurements are one minute apart
time_diff = df["timestamp"].diff()

display(
    time_diff.value_counts().head(10)
)

timestamp
0 days 00:01:00    2075258
Name: count, dtype: int64

In [33]:
# Check for expected 1-minute intervals
expected_interval = pd.Timedelta(minutes=1)

expected_count = (
    time_diff == expected_interval
).sum()

unexpected_count = (
    time_diff != expected_interval
).sum()

print(
    "Expected 1-minute intervals:",
    expected_count
)

print(
    "Unexpected intervals:",
    unexpected_count
)

Expected 1-minute intervals: 2075258
Unexpected intervals: 1


## 10. Find the largest time gaps

In [34]:
gap_report = (
    pd.DataFrame({
        "timestamp": df["timestamp"],
        "time_diff": time_diff
    })
    .sort_values(
        "time_diff",
        ascending=False
    )
)

display(
    gap_report.head(20)
)

,timestamp,time_diff
1,2006-12-16 17:25:00,0 days 00:01:00
1383500,2009-08-03 11:44:00,0 days 00:01:00
1383513,2009-08-03 11:57:00,0 days 00:01:00
1383512,2009-08-03 11:56:00,0 days 00:01:00
1383511,2009-08-03 11:55:00,0 days 00:01:00
1383510,2009-08-03 11:54:00,0 days 00:01:00
1383509,2009-08-03 11:53:00,0 days 00:01:00
1383508,2009-08-03 11:52:00,0 days 00:01:00
1383507,2009-08-03 11:51:00,0 days 00:01:00
1383506,2009-08-03 11:50:00,0 days 00:01:00


In [35]:
# Check negative values
for column in NUMERIC_COLUMNS:
    negative_count = (
        df[column] < 0
    ).sum()

    print(
        f"{column}: {negative_count} negative values"
    )

Global_active_power: 0 negative values
Global_reactive_power: 0 negative values
Voltage: 0 negative values
Global_intensity: 0 negative values
Sub_metering_1: 0 negative values
Sub_metering_2: 0 negative values
Sub_metering_3: 0 negative values


In [36]:
# Check zero values

for column in NUMERIC_COLUMNS:
    zero_count = (
        df[column] == 0
    ).sum()

    print(
        f"{column}: {zero_count} zero values"
    )

Global_active_power: 0 zero values
Global_reactive_power: 481561 zero values
Voltage: 0 zero values
Global_intensity: 0 zero values
Sub_metering_1: 1880175 zero values
Sub_metering_2: 1436830 zero values
Sub_metering_3: 852092 zero values


In [37]:
# Create a missing-data block analysis
df["is_missing_power"] = (
    df["Global_active_power"].isna()
)

In [38]:
df["missing_group"] = (
    df["is_missing_power"]
    != df["is_missing_power"].shift()
).cumsum()

In [39]:
missing_blocks = (
    df[df["is_missing_power"]]
    .groupby("missing_group")
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        missing_count=("timestamp", "count")
    )
    .sort_values(
        "missing_count",
        ascending=False
    )
)

In [40]:
display(
    missing_blocks.head(20)
)

,start_time,end_time,missing_count
missing_group,,,
138,2010-08-17 21:02:00,2010-08-22 21:27:00,7226
140,2010-09-25 03:56:00,2010-09-28 19:12:00,5237
14,2007-04-28 00:21:00,2007-04-30 14:23:00,3723
100,2009-06-13 00:30:00,2009-06-15 07:34:00,3305
118,2010-01-12 14:53:00,2010-01-14 19:01:00,3129
126,2010-03-20 03:52:00,2010-03-21 13:38:00,2027
104,2009-08-13 05:00:00,2009-08-13 19:50:00,891
32,2007-07-15 16:49:00,2007-07-15 18:11:00,83
78,2008-12-10 10:48:00,2008-12-10 11:57:00,70


In [41]:
print(
    "Total missing blocks:",
    len(missing_blocks)
)

Total missing blocks: 71


# 11 — Missing Data Analysis

In [42]:
missing_summary = (
    df.isna()
      .sum()
      .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"]
    / len(df)
    * 100
)

missing_summary = (
    missing_summary
    .sort_values(
        "missing_percentage",
        ascending=False
    )
)

display(missing_summary)

,missing_count,missing_percentage
Global_reactive_power,25979,1.251844
Global_active_power,25979,1.251844
Global_intensity,25979,1.251844
Voltage,25979,1.251844
Sub_metering_3,25979,1.251844
Sub_metering_2,25979,1.251844
Sub_metering_1,25979,1.251844
Time,0,0.000000
Date,0,0.000000
timestamp,0,0.000000


# 12. Calculate missing block duration

In [45]:
missing_blocks["duration"] = (
    missing_blocks["end_time"]
    - missing_blocks["start_time"]
)

missing_blocks["duration"] = (
    missing_blocks["end_time"]
    - missing_blocks["start_time"]
)

display(
    missing_blocks.head(20)
)

,start_time,end_time,missing_count,duration
missing_group,,,,
138,2010-08-17 21:02:00,2010-08-22 21:27:00,7226,5 days 00:25:00
140,2010-09-25 03:56:00,2010-09-28 19:12:00,5237,3 days 15:16:00
14,2007-04-28 00:21:00,2007-04-30 14:23:00,3723,2 days 14:02:00
100,2009-06-13 00:30:00,2009-06-15 07:34:00,3305,2 days 07:04:00
118,2010-01-12 14:53:00,2010-01-14 19:01:00,3129,2 days 04:08:00
126,2010-03-20 03:52:00,2010-03-21 13:38:00,2027,1 days 09:46:00
104,2009-08-13 05:00:00,2009-08-13 19:50:00,891,0 days 14:50:00
32,2007-07-15 16:49:00,2007-07-15 18:11:00,83,0 days 01:22:00
78,2008-12-10 10:48:00,2008-12-10 11:57:00,70,0 days 01:09:00


# 13. Convert duration to hours

In [47]:
missing_blocks["duration_hours"] = (
    missing_blocks["duration"]
    .dt.total_seconds()
    / 3600
)
display(
    missing_blocks[
        [
            "start_time",
            "end_time",
            "missing_count",
            "duration_hours"
        ]
    ].head(20)
)

,start_time,end_time,missing_count,duration_hours
missing_group,,,,
138,2010-08-17 21:02:00,2010-08-22 21:27:00,7226,120.416667
140,2010-09-25 03:56:00,2010-09-28 19:12:00,5237,87.266667
14,2007-04-28 00:21:00,2007-04-30 14:23:00,3723,62.033333
100,2009-06-13 00:30:00,2009-06-15 07:34:00,3305,55.066667
118,2010-01-12 14:53:00,2010-01-14 19:01:00,3129,52.133333
126,2010-03-20 03:52:00,2010-03-21 13:38:00,2027,33.766667
104,2009-08-13 05:00:00,2009-08-13 19:50:00,891,14.833333
32,2007-07-15 16:49:00,2007-07-15 18:11:00,83,1.366667
78,2008-12-10 10:48:00,2008-12-10 11:57:00,70,1.150000


# 14. Categorize the missing blocks

In [52]:
def classify_gap(minutes):
    if minutes <= 5:
        return "very_short"
    elif minutes <= 60:
        return "short"
    elif minutes <= 1440:
        return "medium"
    else:
        return "long"
    
    

In [53]:
missing_blocks["gap_category"] = (
    missing_blocks["missing_count"]
    .apply(classify_gap)
)

In [54]:
display(
    missing_blocks[
        [
            "start_time",
            "end_time",
            "missing_count",
            "duration_hours",
            "gap_category"
        ]
    ]
)

,start_time,end_time,missing_count,duration_hours,gap_category
missing_group,,,,,
138,2010-08-17 21:02:00,2010-08-22 21:27:00,7226,120.416667,long
140,2010-09-25 03:56:00,2010-09-28 19:12:00,5237,87.266667,long
14,2007-04-28 00:21:00,2007-04-30 14:23:00,3723,62.033333,long
100,2009-06-13 00:30:00,2009-06-15 07:34:00,3305,55.066667,long
118,2010-01-12 14:53:00,2010-01-14 19:01:00,3129,52.133333,long
...,...,...,...,...,...
130,2010-05-13 21:05:00,2010-05-13 21:05:00,1,0.000000,very_short
134,2010-06-29 16:53:00,2010-06-29 16:53:00,1,0.000000,very_short
132,2010-06-12 17:35:00,2010-06-12 17:35:00,1,0.000000,very_short


# 15. Count each category

In [55]:
gap_distribution = (
    missing_blocks["gap_category"]
    .value_counts()
)

display(gap_distribution)

gap_category
very_short    55
short          7
long           6
medium         3
Name: count, dtype: int64

In [56]:
missing_blocks["duration"] = (
    missing_blocks["end_time"]
    - missing_blocks["start_time"]
)

missing_blocks["duration_hours"] = (
    missing_blocks["duration"]
    .dt.total_seconds() / 3600
)

In [57]:
missing_blocks["gap_category"] = (
    missing_blocks["missing_count"]
    .apply(classify_gap)
)

display(
    missing_blocks[
        [
            "start_time",
            "end_time",
            "missing_count",
            "duration_hours",
            "gap_category"
        ]
    ]
)

,start_time,end_time,missing_count,duration_hours,gap_category
missing_group,,,,,
138,2010-08-17 21:02:00,2010-08-22 21:27:00,7226,120.416667,long
140,2010-09-25 03:56:00,2010-09-28 19:12:00,5237,87.266667,long
14,2007-04-28 00:21:00,2007-04-30 14:23:00,3723,62.033333,long
100,2009-06-13 00:30:00,2009-06-15 07:34:00,3305,55.066667,long
118,2010-01-12 14:53:00,2010-01-14 19:01:00,3129,52.133333,long
...,...,...,...,...,...
130,2010-05-13 21:05:00,2010-05-13 21:05:00,1,0.000000,very_short
134,2010-06-29 16:53:00,2010-06-29 16:53:00,1,0.000000,very_short
132,2010-06-12 17:35:00,2010-06-12 17:35:00,1,0.000000,very_short


# 16. First, analyze the gap distribution

In [58]:
gap_distribution = (
    missing_blocks["gap_category"]
    .value_counts()
    .reindex(
        ["very_short", "short", "medium", "long"],
        fill_value=0
    )
)

display(gap_distribution)

gap_category
very_short    55
short          7
medium         3
long           6
Name: count, dtype: int64

In [59]:
missing_by_category = ( 
    missing_blocks
    .groupby("gap_category")["missing_count"]
    .sum()
    .reindex(
        ["very_short", "short", "medium", "long"],
        fill_value=0
    )
)

display(missing_by_category)

gap_category
very_short       76
short           212
medium         1044
long          24647
Name: missing_count, dtype: int64

# 17. Calculate the percentage of missing records by category

In [60]:
missing_by_category_percentage = (
    missing_by_category
    / df["Sub_metering_3"].isna().sum()
    * 100
)

missing_by_category_percentage = (
    missing_by_category_percentage
    .round(2)
)

display(missing_by_category_percentage)

gap_category
very_short     0.29
short          0.82
medium         4.02
long          94.87
Name: missing_count, dtype: float64

# 18. Implement the missing-data strategy

In [61]:
df_processed = df.copy()

print("Original shape:", df.shape)
print("Processing copy created:", df_processed.shape)

Original shape: (2075259, 12)
Processing copy created: (2075259, 12)


# 19. Create a missing-value flag

In [63]:
df_processed["Sub_metering_3_was_missing"] = ( # 
    df_processed["Sub_metering_3"].isna()
)
print(
    "Original missing values:",
    df_processed["Sub_metering_3_was_missing"].sum()
)

Original missing values: 25979


# 20. Create the gap ID again

In [64]:
df_processed["is_missing_sub3"] = (
    df_processed["Sub_metering_3"].isna()
)

df_processed["missing_group"] = (
    df_processed["is_missing_sub3"]
    != df_processed["is_missing_sub3"].shift()
).cumsum()

# 21. Calculate each missing block length

In [65]:
missing_block_sizes = (
    df_processed[
        df_processed["is_missing_sub3"]
    ]
    .groupby("missing_group")
    .size()
)

In [66]:
print(
    missing_block_sizes.describe()
)

count      71.000000
mean      365.901408
std      1251.468043
min         1.000000
25%         1.000000
50%         1.000000
75%         3.000000
max      7226.000000
dtype: float64


# 22. Identify short gaps

In [67]:
SHORT_GAP_LIMIT = 60

In [68]:
short_missing_groups = (
    missing_block_sizes[
        missing_block_sizes <= SHORT_GAP_LIMIT
    ]
    .index
)

In [69]:
print(
    "Short missing groups:",
    len(short_missing_groups)
)

Short missing groups: 62


# 23.Create a mask for values we are allowed to interpolate

In [71]:
short_gap_mask = (
    df_processed["missing_group"]
    .isin(short_missing_groups)
    & df_processed["is_missing_sub3"]
)
print(
    "Values eligible for interpolation:",
    short_gap_mask.sum()
)

Values eligible for interpolation: 288


# 24. Interpolate only those short gaps

In [72]:
interpolated_sub3 = (
    df_processed["Sub_metering_3"]
    .interpolate(
        method="linear",
        limit=SHORT_GAP_LIMIT,
        limit_direction="both"
    )
)

In [73]:
df_processed.loc[
    short_gap_mask,
    "Sub_metering_3"
] = interpolated_sub3.loc[
    short_gap_mask
]

In [ ]:
remaining_missing = ( 
    df_processed["Sub_metering_3"].isna().sum()
)

print(
    "Remaining Sub_metering_3 missing values:",
    remaining_missing
)

Remaining Sub_metering_3 missing values: 25691


# 25. Verify that long gaps remain missing

In [75]:
largest_group = (
    missing_block_sizes
    .sort_values(ascending=False)
    .index[0]
)

largest_gap = df_processed[
    df_processed["missing_group"] == largest_group
]

display(
    largest_gap[
        [
            "timestamp",
            "Sub_metering_3"
        ]
    ].head()
)

,timestamp,Sub_metering_3
1929818,2010-08-17 21:02:00,NaN
1929819,2010-08-17 21:03:00,NaN
1929820,2010-08-17 21:04:00,NaN
1929821,2010-08-17 21:05:00,NaN
1929822,2010-08-17 21:06:00,NaN


# 26. Remove temporary columns

In [76]:
df_processed = df_processed.drop(
    columns=[
        "is_missing_sub3",
        "missing_group"
    ]
)

# 27. Check the final data structure

In [77]:
print(
    df_processed.shape
)

print(
    df_processed.columns.tolist()
)

(2075259, 12)
['Date', 'Time', 'Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3', 'timestamp', 'is_missing_power', 'Sub_metering_3_was_missing']


# 28. Verify the target variable

In [78]:
print(
    "Global_active_power missing:",
    df_processed["Global_active_power"].isna().sum()
)

Global_active_power missing: 25979


In [79]:
print("Missing values after numeric conversion:")
display(
    df[NUMERIC_COLUMNS].isna().sum()
)

Missing values after numeric conversion:


Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64

In [80]:
missing_after_conversion = (
    df[NUMERIC_COLUMNS]
    .isna()
    .sum()
    .to_frame("missing_count")
)

missing_after_conversion["missing_percentage"] = (
    missing_after_conversion["missing_count"]
    / len(df)
    * 100
)

display(
    missing_after_conversion
    .sort_values(
        "missing_count",
        ascending=False
    )
)

,missing_count,missing_percentage
Global_active_power,25979,1.251844
Global_reactive_power,25979,1.251844
Voltage,25979,1.251844
Global_intensity,25979,1.251844
Sub_metering_1,25979,1.251844
Sub_metering_2,25979,1.251844
Sub_metering_3,25979,1.251844


In [81]:
# Check whether all missing measurements occur together
missing_pattern = (
    df[NUMERIC_COLUMNS]
    .isna()
    .value_counts()
)

display(missing_pattern)

Global_active_power  Global_reactive_power  Voltage  Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3
False                False                  False    False             False           False           False             2049280
True                 True                   True     True              True            True            True                25979
Name: count, dtype: int64

In [82]:
all_measurements_missing = (
    df[NUMERIC_COLUMNS]
    .isna()
    .all(axis=1)
)

print(
    "Rows where ALL measurements are missing:",
    all_measurements_missing.sum()
)

Rows where ALL measurements are missing: 25979


In [83]:
partial_missing = (
    df[NUMERIC_COLUMNS]
    .isna()
    .any(axis=1)
    &
    ~all_measurements_missing
)

print(
    "Rows with PARTIAL missing measurements:",
    partial_missing.sum()
)

Rows with PARTIAL missing measurements: 0


In [84]:
# Check the first missing period
display(
    df[
        all_measurements_missing
    ][
        [
            "timestamp",
            *NUMERIC_COLUMNS
        ]
    ].head(20)
)

,timestamp,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
6839,2006-12-21 11:23:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6840,2006-12-21 11:24:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19724,2006-12-30 10:08:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19725,2006-12-30 10:09:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41832,2007-01-14 18:36:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
61909,2007-01-28 17:13:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98254,2007-02-22 22:58:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98255,2007-02-22 22:59:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
142588,2007-03-25 17:52:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
190497,2007-04-28 00:21:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
# Check your missing blocks again
df["Sub_metering_3"].isna()

0          False
1          False
2          False
3          False
4          False
           ...  
2075254    False
2075255    False
2075256    False
2075257    False
2075258    False
Name: Sub_metering_3, Length: 2075259, dtype: bool

In [86]:
df["all_measurements_missing"] = (
    df[NUMERIC_COLUMNS]
    .isna()
    .all(axis=1)
)

In [87]:
df["missing_group"] = (
    df["all_measurements_missing"]
    != df["all_measurements_missing"].shift()
).cumsum()

In [88]:
measurement_missing_blocks = (
    df[
        df["all_measurements_missing"]
    ]
    .groupby("missing_group")
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        missing_count=("timestamp", "count")
    )
    .sort_values(
        "missing_count",
        ascending=False
    )
)

In [89]:
display(
    measurement_missing_blocks.head(20)
)

,start_time,end_time,missing_count
missing_group,,,
138,2010-08-17 21:02:00,2010-08-22 21:27:00,7226
140,2010-09-25 03:56:00,2010-09-28 19:12:00,5237
14,2007-04-28 00:21:00,2007-04-30 14:23:00,3723
100,2009-06-13 00:30:00,2009-06-15 07:34:00,3305
118,2010-01-12 14:53:00,2010-01-14 19:01:00,3129
126,2010-03-20 03:52:00,2010-03-21 13:38:00,2027
104,2009-08-13 05:00:00,2009-08-13 19:50:00,891
32,2007-07-15 16:49:00,2007-07-15 18:11:00,83
78,2008-12-10 10:48:00,2008-12-10 11:57:00,70


In [90]:
display(
    df[NUMERIC_COLUMNS].isna().sum()
)

Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64

In [91]:
print(
    "Rows where ALL measurements are missing:",
    df[NUMERIC_COLUMNS].isna().all(axis=1).sum()
)

print(
    "Rows with PARTIAL missing measurements:",
    (
        df[NUMERIC_COLUMNS].isna().any(axis=1)
        &
        ~df[NUMERIC_COLUMNS].isna().all(axis=1)
    ).sum()
)

Rows where ALL measurements are missing: 25979
Rows with PARTIAL missing measurements: 0


In [92]:
missing_pattern = (
    df[NUMERIC_COLUMNS]
    .isna()
    .value_counts()
)

display(missing_pattern)

Global_active_power  Global_reactive_power  Voltage  Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3
False                False                  False    False             False           False           False             2049280
True                 True                   True     True              True            True            True                25979
Name: count, dtype: int64

In [94]:
#  Remove the previous incorrect interpolation code
interpolated_sub3 = (
    df_processed["Sub_metering_3"]
    .interpolate(method="linear")
)

In [95]:
df_processed = df.copy()

print("Processed dataset created.")
print("Shape:", df_processed.shape)

Processed dataset created.
Shape: (2075259, 13)


# 29. Create the measurement-outage flag

In [96]:
df_processed["is_measurement_missing"] = (
    df_processed[NUMERIC_COLUMNS]
    .isna()
    .all(axis=1)
)

In [97]:
print(
    "Measurement outage rows:",
    df_processed["is_measurement_missing"].sum()
)

Measurement outage rows: 25979


In [98]:
# Create a target availability flag

df_processed["target_available"] = (
    df_processed["Global_active_power"].notna()
)

In [99]:
print(
    "Target available:",
    df_processed["target_available"].sum()
)

print(
    "Target missing:",
    df_processed["target_available"].eq(False).sum()
)

Target available: 2049280
Target missing: 25979


# 30. Verify the relationship

In [100]:
print(
    "Measurement missing but target available:",
    (
        df_processed["is_measurement_missing"]
        &
        df_processed["target_available"]
    ).sum()
)

Measurement missing but target available: 0


# 31. Create the missing block information

In [101]:
df_processed["missing_group"] = (
    df_processed["is_measurement_missing"]
    != df_processed["is_measurement_missing"].shift()
).cumsum()

In [102]:
missing_blocks = (
    df_processed[
        df_processed["is_measurement_missing"]
    ]
    .groupby("missing_group")
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        missing_count=("timestamp", "count")
    )
    .sort_values(
        "missing_count",
        ascending=False
    )
)

In [103]:
missing_blocks["duration_minutes"] = (
    missing_blocks["missing_count"]
)

In [104]:
missing_blocks["duration_hours"] = (
    missing_blocks["duration_minutes"] / 60
)

In [105]:
display(
    missing_blocks.head(20)
)

,start_time,end_time,missing_count,duration_minutes,duration_hours
missing_group,,,,,
138,2010-08-17 21:02:00,2010-08-22 21:27:00,7226,7226,120.433333
140,2010-09-25 03:56:00,2010-09-28 19:12:00,5237,5237,87.283333
14,2007-04-28 00:21:00,2007-04-30 14:23:00,3723,3723,62.050000
100,2009-06-13 00:30:00,2009-06-15 07:34:00,3305,3305,55.083333
118,2010-01-12 14:53:00,2010-01-14 19:01:00,3129,3129,52.150000
126,2010-03-20 03:52:00,2010-03-21 13:38:00,2027,2027,33.783333
104,2009-08-13 05:00:00,2009-08-13 19:50:00,891,891,14.850000
32,2007-07-15 16:49:00,2007-07-15 18:11:00,83,83,1.383333
78,2008-12-10 10:48:00,2008-12-10 11:57:00,70,70,1.166667


# 32. Create a gap category

In [106]:
def classify_gap(minutes):
    if minutes <= 5:
        return "very_short"
    elif minutes <= 60:
        return "short"
    elif minutes <= 1440:
        return "medium"
    else:
        return "long"

In [107]:
missing_blocks["gap_category"] = (
    missing_blocks["duration_minutes"]
    .apply(classify_gap)
)

In [108]:
display(
    missing_blocks[
        [
            "start_time",
            "end_time",
            "missing_count",
            "duration_minutes",
            "duration_hours",
            "gap_category"
        ]
    ]
)

,start_time,end_time,missing_count,duration_minutes,duration_hours,gap_category
missing_group,,,,,,
138,2010-08-17 21:02:00,2010-08-22 21:27:00,7226,7226,120.433333,long
140,2010-09-25 03:56:00,2010-09-28 19:12:00,5237,5237,87.283333,long
14,2007-04-28 00:21:00,2007-04-30 14:23:00,3723,3723,62.050000,long
100,2009-06-13 00:30:00,2009-06-15 07:34:00,3305,3305,55.083333,long
118,2010-01-12 14:53:00,2010-01-14 19:01:00,3129,3129,52.150000,long
...,...,...,...,...,...,...
130,2010-05-13 21:05:00,2010-05-13 21:05:00,1,1,0.016667,very_short
134,2010-06-29 16:53:00,2010-06-29 16:53:00,1,1,0.016667,very_short
132,2010-06-12 17:35:00,2010-06-12 17:35:00,1,1,0.016667,very_short


# 33. Create a missing-data summary

In [109]:
missing_summary = {
    "total_rows": len(df_processed),
    "complete_measurement_rows": int(
        (~df_processed["is_measurement_missing"]).sum()
    ),
    "measurement_outage_rows": int(
        df_processed["is_measurement_missing"].sum()
    ),
    "measurement_outage_percentage": round(
        df_processed["is_measurement_missing"].mean() * 100,
        2
    ),
    "missing_blocks": len(missing_blocks),
    "target": "Global_active_power"
}

missing_summary

{'total_rows': 2075259,
 'complete_measurement_rows': 2049280,
 'measurement_outage_rows': 25979,
 'measurement_outage_percentage': np.float64(1.25),
 'missing_blocks': 71,
 'target': 'Global_active_power'}

# 34. Now remove only the temporary grouping column

In [110]:
df_processed.drop(
    columns=["missing_group"],
    inplace=True
)

In [111]:
# Verify the processed dataset
print("========== FINAL DATA CHECK ==========")

print("Rows:", len(df_processed))

print(
    "Columns:",
    len(df_processed.columns)
)

print(
    "Measurement outage rows:",
    df_processed["is_measurement_missing"].sum()
)

print(
    "Target missing:",
    df_processed["Global_active_power"].isna().sum()
)

print(
    "Duplicate timestamps:",
    df_processed["timestamp"].duplicated().sum()
)

print(
    "Timestamp sorted:",
    df_processed["timestamp"].is_monotonic_increasing
)

print(
    "Invalid timestamps:",
    df_processed["timestamp"].isna().sum()
)

========== FINAL DATA CHECK ==========
Rows: 2075259
Columns: 14
Measurement outage rows: 25979
Target missing: 25979
Duplicate timestamps: 0
Timestamp sorted: True
Invalid timestamps: 0


In [112]:
RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "data_quality"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [113]:
missing_blocks.to_csv(
    RESULTS_DIR / "missing_blocks.csv"
)

In [114]:
print(
    "Missing-block report saved to:",
    RESULTS_DIR / "missing_blocks.csv"
)

Missing-block report saved to: c:\Users\sayum\Desktop\smartenergy-ai\results\data_quality\missing_blocks.csv


In [115]:
import json

In [116]:
validation_summary = {
    "dataset": "household_power_consumption",
    "total_rows": int(len(df_processed)),
    "total_columns": int(len(df_processed.columns)),
    "complete_measurement_rows": int(
        (~df_processed["is_measurement_missing"]).sum()
    ),
    "measurement_outage_rows": int(
        df_processed["is_measurement_missing"].sum()
    ),
    "measurement_outage_percentage": round(
        df_processed["is_measurement_missing"].mean() * 100,
        2
    ),
    "missing_blocks": int(len(missing_blocks)),
    "duplicate_timestamps": int(
        df_processed["timestamp"].duplicated().sum()
    ),
    "invalid_timestamps": int(
        df_processed["timestamp"].isna().sum()
    ),
    "target": "Global_active_power"
}

In [118]:
with open(
    RESULTS_DIR / "validation_summary.json",
    "w"
) as file:
    json.dump(
        validation_summary,
        file,
        indent=4
    )

In [123]:
# Run the preprocessing pipeline
df_processed = preprocess_data(df)

c:\Users\sayum\Desktop\smartenergy-ai\src\data\preprocessing.py:22: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["timestamp"] = pd.to_datetime(


In [124]:
print(
    "Processed dataset shape:",
    df_processed.shape
)

Processed dataset shape: (2075259, 12)


In [125]:
# Run validation
validation_results = validate_dataset(
    df_processed
)

validation_results

{'rows': 2075259,
 'columns': 12,
 'invalid_timestamps': 0,
 'duplicate_timestamps': 0,
 'timestamp_sorted': True,
 'measurement_outage_rows': 25979,
 'measurement_outage_percentage': np.float64(1.25),
 'target_missing': 25979}

In [126]:
# Important test
assert validation_results["invalid_timestamps"] == 0

assert validation_results["duplicate_timestamps"] == 0

assert validation_results["timestamp_sorted"] is True

assert validation_results["measurement_outage_rows"] == 25979

assert validation_results["target_missing"] == 25979

print("All validation assertions passed.")

All validation assertions passed.
